In [1]:
from pathlib import Path
import numpy as np
import Century_APEX as md
import gc

In [2]:
# Select sites
SITES = ["ZAN"]

nyears = 2000
dt = 1 / 24
ndays = nyears * 365
nsim = int(ndays / dt)

for SITE in SITES:

    print(f"\n{'='*50}")
    print(f"Running {SITE}")
    print(f"{'='*50}", flush=True)

    parameters = md.SITES[SITE]

    # Rainfall
    rain = md.poisson_process(n=nsim, alpha=parameters["alpha"], lbd=parameters["lbd"] * dt)

    # NPP
    days = np.arange(365)
    shape = np.sin(np.pi * days / 365)
    daily_npp = parameters["annual_avg_npp"] * shape / shape.sum()
    hourly_npp = np.repeat(daily_npp, 24)
    npp = np.tile(hourly_npp, nyears)

    kstex = parameters["kstex"]
    kstot = md.kstot(parameters["SAN"], kstex)
    phitex = parameters["phitex"]
    phitot = md.phitot(parameters["SAN"], phitex)

    # Run model
    sim = md.run_julia(rain, npp, kstex, kstot, phitex, phitot, site=SITE)

    # Annual averages
    sim_year = sim.pivot_table(values=["s", "StrLitter", "MetLitter", "ACTIVE", "SLOW", "PASSIVE", "NPP_c", "Fa", "Fp", "Fs", "Fls", "Flm", "npp_",
                                       "CS", "OX", "X1", "Bulk Density", "Bulk Density Plow", "WBMX", "F", "X3", "RSD", "Fi"], index="Years")

    print(sim_year)

    # Save results
    for filename in parameters["output_files"]:
        Path(filename).parent.mkdir(parents=True, exist_ok=True)
        sim_year.to_csv(filename, index=True, header=True)
        print("Saved:", filename, flush=True)

    print(f"Completed {SITE}", flush=True)

    # Clear memory before next site
    del rain, npp, sim, sim_year
    gc.collect()

print("\nAll eight sites completed!")


Running ZAN


100%|████████████████████████████████████████████████████████████████████| 17520000/17520000 [57:44<00:00, 5057.65it/s]


          ACTIVE  Bulk Density  Bulk Density Plow        CS         F  \
Years                                                                   
1      41.287952      1.601485           1.601485  0.000000  0.000000   
2      10.413297      1.607348           1.442988  2.083643  0.000941   
3       8.574617      1.614574           1.404279  2.805988  0.000935   
4       7.246636      1.622547           1.378983  3.298645  0.000936   
5       6.403184      1.629898           1.381680  3.430110  0.000940   
...          ...           ...                ...       ...       ...   
1996   19.218279      1.696870           1.696870  0.944814  0.024843   
1997   19.404932      1.696932           1.696932  0.936261  0.023059   
1998   19.339944      1.696944           1.696944  0.943482  0.022520   
1999   19.432472      1.696951           1.696951  0.919297  0.027034   
2000   19.875964      1.696769           1.696769  0.932075  0.026124   

             Fa        Fi       Flm       Fls     